In [9]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from datetime import datetime

SCHEDULE_PATH = Path("baseball_schedule.csv")

SCHEMA = {
    "date": "datetime64[ns]",
    "home_team": "string",
    "away_team": "string",
    "time": "string",   # HH:MM
    "venue": "string"
}

TEAMS = ["Brothers", "Lions", "Hawks", "Dragons", "Guardians", "Monkeys"]
VENUES = [
    "Big-Dome - Taipei", "Inter - TaiChong", "TianMu - Taipei", "ShinJuan - New Taipei",
    "Rakuten - Taoyuan", "LungTan - Taoyuan", "Chengching Lake- KaoShong"
]
MEMBER_HAWKS = [ "🥭", "🐷" ]
MEMBER_BROTHERS = [ "🍞", "🐗" ]
MEMBER_LIONS = [ "Mingo", "An Ji-hyun" ]
MEMBER_DRAGONS = [ "Mingo", "An Ji-hyun" ]
MEMBER_MONKEYSS = [ "Mingo", "An Ji-hyun" ]
MEMBER_GUARDIANS = [ "Mingo", "An Ji-hyun" ]

def _empty_df() -> pd.DataFrame:
    return pd.DataFrame(columns=SCHEMA.keys()).astype(SCHEMA)

def load_schedule() -> pd.DataFrame:
    if SCHEDULE_PATH.exists():
        df = pd.read_csv(SCHEDULE_PATH)
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"]).dt.normalize()
        for col, dtype in SCHEMA.items():
            if col != "date":
                if col not in df.columns:
                    df[col] = pd.Series(dtype=dtype)
                else:
                    df[col] = df[col].astype("string")
        return df.sort_values(["date","time","home_team","away_team"], na_position="last").reset_index(drop=True)
    return _empty_df()

def save_schedule(df: pd.DataFrame) -> None:
    df = df[list(SCHEMA.keys())].copy()
    df.to_csv(SCHEDULE_PATH, index=False)

def clear_schedule(backup: bool = True):
    global schedule_df
    backup_path = None
    if backup and SCHEDULE_PATH.exists():
        ts = datetime.now().strftime("%Y%m%d-%H%M%S")
        backup_path = SCHEDULE_PATH.with_name(f"{SCHEDULE_PATH.stem}.backup-{ts}{SCHEDULE_PATH.suffix}")
        schedule_df.copy().to_csv(backup_path, index=False)
    schedule_df = _empty_df()
    save_schedule(schedule_df)
    return backup_path

# ---- In-memory table ----
schedule_df = load_schedule()

# ---- CRUD / viewers ----
def add_game(home_team: str, away_team: str, dt, time_str: str | None = None, venue: str | None = None):
    """Insert/update a game (unique by date+home+away). Returns rows for that date."""
    global schedule_df
    if not home_team or not away_team:
        raise ValueError("home_team and away_team are required.")
    if home_team == away_team:
        raise ValueError("home_team and away_team cannot be the same.")

    date_norm = pd.to_datetime(dt).normalize()
    time_str = None if (time_str is None or str(time_str).strip() == "") else str(time_str)
    venue = None if (venue is None or str(venue).strip() == "") else str(venue)

    dup = (
        (schedule_df["date"] == date_norm) &
        (schedule_df["home_team"].str.casefold() == home_team.casefold()) &
        (schedule_df["away_team"].str.casefold() == away_team.casefold())
    )
    if dup.any():
        idx = schedule_df[dup].index[0]
        if time_str is not None:
            schedule_df.at[idx, "time"] = time_str
        if venue is not None:
            schedule_df.at[idx, "venue"] = venue
    else:
        schedule_df = pd.concat([schedule_df, pd.DataFrame([{
            "date": date_norm,
            "home_team": home_team,
            "away_team": away_team,
            "time": time_str,
            "venue": venue,
        }])], ignore_index=True)

    schedule_df = schedule_df.sort_values(["date","time","home_team","away_team"], na_position="last").reset_index(drop=True)
    save_schedule(schedule_df)
    return schedule_df[schedule_df["date"] == date_norm]

def view_all():
    return schedule_df.sort_values(["date","time","home_team","away_team"], na_position="last").reset_index(drop=True)

def view_date(d):
    date_norm = pd.to_datetime(d).normalize()
    return schedule_df[schedule_df["date"] == date_norm].sort_values(["time","home_team","away_team"], na_position="last").reset_index(drop=True)

In [14]:
# ---- Small helper ----
def _time_to_str(t):
    if t is None: return None
    return f"{t.hour:02d}:{t.minute:02d}"

# ---- Tab 1: Add Game ----
date_picker = widgets.DatePicker(description="Date", value=None)
home_dd     = widgets.Dropdown(options=TEAMS, description="Home")
away_dd     = widgets.Dropdown(options=TEAMS, description="Away")
time_picker = widgets.TimePicker(description="Time", value=None)
venue_dd    = widgets.Dropdown(options=VENUES, description="Venue")
send_btn    = widgets.Button(description="Send", button_style="primary")
out_add     = widgets.Output()

def on_send_clicked(_):
    with out_add:
        clear_output(wait=True)

        picked_date = date_picker.value
        home, away  = home_dd.value, away_dd.value
        time_str    = _time_to_str(time_picker.value)
        venue       = venue_dd.value

        errors = []
        if picked_date is None:
            errors.append("Please pick a date.")
        if home == away:
            errors.append("Home and Away cannot be the same team.")
        if venue not in VENUES:
            errors.append("Please choose a valid venue.")
        if errors:
            print("⚠️ Validation failed:")
            for e in errors: print(" -", e)
            return

        day_df = add_game(home, away, picked_date, time_str, venue)

        print("✅ Saved to schedule.")
        print(f"  Date : {pd.to_datetime(picked_date).date()}")
        print(f"  Home : {home}")
        print(f"  Away : {away}")
        print(f"  Time : {time_str or '(no time)'}")
        print(f"  Venue: {venue}")
        print("\nDay schedule:")
        display(day_df.reset_index(drop=True))

send_btn.on_click(on_send_clicked)
tab_add_game = widgets.VBox([date_picker, home_dd, away_dd, time_picker, venue_dd, send_btn, out_add])

# ---- Tab 2: Add Players (placeholder; we will implement next) ----

TEAM_TO_MEMBERS = {
    "Hawks": MEMBER_HAWKS,
    "Brothers": MEMBER_BROTHERS,
}


default_team = None if not TEAMS else TEAMS[0]

team_dd = widgets.Dropdown(
    options=TEAMS,
    value=default_team,
    description="Team",
)

members_box = widgets.VBox([])
members_info = widgets.Output()

def get_members_for_team(team_name: str):
    return list(TEAM_TO_MEMBERS.get(team_name, []))

def rebuild_member_checkboxes(team: str):
    members = get_members_for_team(team)
    checkboxes = [widgets.Checkbox(description=m, value=False, indent=False) for m in members]
    members_box.children = checkboxes
    with members_info:
        clear_output(wait=True)
        print(f"Team: {team} | Members: {len(members)}")

def on_team_change(change):
    if change["name"] == "value":
        rebuild_member_checkboxes(change["new"])

team_dd.observe(on_team_change, names="value")
rebuild_member_checkboxes(team_dd.value)

tab_add_member = widgets.VBox([
    widgets.HTML("<b>Add Member</b>"),
    team_dd,
    members_box,
    members_info,
])

# ---- Assemble Tabs ----
tabs = widgets.Tab(children=[tab_add_game, tab_add_member])
tabs.set_title(0, "Add Game")
tabs.set_title(1, "Add Member")
display(tabs)

In [7]:
# Clean all schedule data

clear_schedule(backup=False)

In [11]:
# View Data

view_all()
#view_date("2025-09-25")

,date,home_team,away_team,time,venue
